<a href="https://colab.research.google.com/github/Wajahat725/EchoLock/blob/main/echolock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Introduction


# 🛡️ EchoLock: Improved Session Password-Based Security System

**Objective:** To train a Machine Learning model that detects if an attacker can deduce a user's secret sequence by observing multiple login sessions.

---
### 📌 Methodology
1. **Data Loading:** Load session logs and user secrets.
2. **Preprocessing:** Filter successful sessions and aggregate observations for the first 5 sessions.
3. **Feature Engineering:** Calculate frequency and count of each observed symbol per click rank.
4. **Model Training:** Train a Logistic Regression classifier to predict if the observed symbol is the actual secret.
5. **Evaluation:** Assess model accuracy against a majority-vote baseline.

 Import Libraries & Setup

In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Global Configurations
N_SESSIONS = 5
SYMBOL_POOL = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")

print("Libraries imported and setup completed.")

Data Loading

In [ ]:
# 2. Load the Datasets
try:
    sessions = pd.read_csv("login_sessions.csv")
    secrets = pd.read_csv("secrets.csv")

    print(f"Data Loaded Successfully!")
    print(f"Sessions Dataset Shape: {sessions.shape}")
    print(f"Secrets Dataset Shape:  {secrets.shape}")

    display(sessions.head())
except FileNotFoundError:
    print("Error: Please upload 'login_sessions.csv' and 'secrets.csv' to the Colab environment.")

Data Preprocessing & Feature Engineering

In [ ]:
# 3. Data Preprocessing
# Keep only successful sessions and sort them chronologically
sessions = sessions[sessions.session_success == True].sort_values(
    ["user_id", "timestamp", "click_rank"]
)

# Create a lookup for user secrets
secret_lookup = dict(zip(secrets.user_id, secrets.secret_sequence))

rows = []

# Iterate through each user's observations per click rank
for (user_id, rank), group in sessions.groupby(["user_id", "click_rank"]):
    # Limit observation to N_SESSIONS
    obs = group["attacker_observed_symbol"].astype(str).tolist()[:N_SESSIONS]
    obs = [o for o in obs if o not in ("", "nan")]
    total_obs = len(obs)

    if total_obs == 0:
        continue

    counts = {s: obs.count(s) for s in set(obs)}
    secret_seq = secret_lookup.get(user_id)

    if not secret_seq:
        continue

    true_symbol = secret_seq[rank - 1] if rank - 1 < len(secret_seq) else None

    # Create features for every possible candidate symbol
    for cand in SYMBOL_POOL:
        c = counts.get(cand, 0)
        rows.append({
            "user_id": user_id,
            "click_rank": rank,
            "secret_length": len(secret_seq),
            "candidate": cand,
            "count": c,
            "total_obs": total_obs,
            "freq": c / total_obs,
            "label": int(cand == true_symbol), # 1 if it's the real secret, 0 otherwise
        })

feat_df = pd.DataFrame(rows)
print(f"Feature Engineering Complete. Dataset Shape: {feat_df.shape}")
display(feat_df.head())

Train-Test Split

In [ ]:
# 4. Train-Test Split (Grouped by User to prevent data leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(feat_df, groups=feat_df["user_id"]))

train_df = feat_df.iloc[train_idx]
test_df = feat_df.iloc[test_idx]

FEATURES = ["click_rank", "secret_length", "count", "total_obs", "freq"]

X_train, y_train = train_df[FEATURES], train_df["label"]
X_test, y_test = test_df[FEATURES], test_df["label"]

print(f"Data Split Complete.")
print(f"Train Users: {train_df.user_id.nunique()} | Test Users: {test_df.user_id.nunique()}")
print(f"Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}")

Model Training

In [ ]:
# 5. Train Logistic Regression Model
# Using class_weight='balanced' because class 0 (not secret) is much larger than class 1 (secret)
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully.")

 Evaluation & Results

In [23]:
# 6. Evaluate the Model
test_df = test_df.copy()
test_df["pred_proba"] = model.predict_proba(X_test)[:, 1]

correct_model, correct_baseline, total_groups = 0, 0, 0

# Evaluate how well the model identifies the secret compared to a simple majority-vote
for (user_id, rank), g in test_df.groupby(["user_id", "click_rank"]):
    total_groups += 1

    # Model's pick (highest probability)
    model_pick = g.loc[g["pred_proba"].idxmax(), "candidate"]

    # Baseline pick (most frequently observed)
    baseline_pick = g.loc[g["count"].idxmax(), "candidate"]

    # Actual secret
    true_pick = g.loc[g["label"] == 1, "candidate"]
    true_pick = true_pick.iloc[0] if len(true_pick) else None

    correct_model += int(model_pick == true_pick)
    correct_baseline += int(baseline_pick == true_pick)

print("="*50)
print(f"AI LEAKAGE DETECTION RESULTS (N_SESSIONS = {N_SESSIONS})")
print("="*50)
print(f"Majority-Vote Baseline Accuracy : {correct_baseline/total_groups:.3f}")
print(f"Trained LogisticRegression Acc  : {correct_model/total_groups:.3f}")
print(f"Evaluated on {total_groups} user/rank groups.")
print("="*50)

AI LEAKAGE DETECTION RESULTS (N_SESSIONS = 5)
Majority-Vote Baseline Accuracy : 0.957
Trained LogisticRegression Acc  : 0.957
Evaluated on 70 user/rank groups.
